# Day 10 — Testing with pytest
Objectives:
- Write unit tests and run pytest.
- Use assertions and parametrize.
- Structure tests for utilities created earlier.

<!-- BEGIN BEGINNER NOTEBOOK DEEP DIVE -->
## How to use this notebook

This is the editable learner artifact for `python-10`. Read
`python/ds-60day/companion-guides/day10_testing_pytest.md` first, then work here with the **Python (ds60sqlpy)**
kernel. Restart the kernel and run from top to bottom so an earlier
hidden value cannot make later code appear correct.

For every example: (1) write a prediction, (2) run the cell,
(3) compare the exact value, type, shape, rows, or side effect with
the stated observation, and (4) explain one mismatch before moving
on. For every exercise, use its dedicated work cell and include a
real assertion or bounded inspection. The official solution stays
closed until you have a tested attempt.

The notebook is deliberately offline after course setup. Do not add
`%pip`, credentials, absolute developer paths, or shell-specific
setup here. If an import fails, use the repository doctor and the
catalog from a terminal rather than changing only this kernel.

## Core mental model

A test is a small executable example of observable behavior. Arrange
the inputs, act once, then assert the output or failure. Tests do not
prove a program has no bugs; they make chosen contracts repeatable and
reveal when later changes violate them.

Good cases cover a normal path, exact boundaries, and invalid input.
Keep each test focused enough that its name and failure identify the
broken behavior. Fixtures own reusable setup and teardown. Parametrized
tests express the same contract across several inputs without hiding
which case failed.

### Vocabulary

- **test case:** one named example with inputs and expected behavior.
- **assertion:** a claim that must be true for the test to pass.
- **fixture:** reusable setup data or a managed resource supplied to tests.
- **parametrization:** running one test contract with multiple named inputs.
- **regression:** behavior that used to work but was later broken.
- **test isolation:** the property that one test does not depend on another's state or order.

## Syntax anatomy

In `assert actual == expected`, pytest can display both values and their
difference. `with pytest.raises(ValueError, match="positive"):` states
that the indented call must raise that exact failure and optionally
match a stable message fragment. The test name should state behavior,
such as `test_safe_divide_rejects_zero_denominator`.

### Worked example 1 — Test a pure calculation with plain assertions

Direct equality makes failures explain the two values. Before running the next cell, predict its final displayed
value and identify the line responsible for every intermediate.

In [ ]:
def subtotal(prices: list[float]) -> float:
    return sum(prices)

assert subtotal([2.5, 3.0]) == 5.5
assert subtotal([]) == 0
"two subtotal contracts passed"

**Expected observation:** `'two subtotal contracts passed'`. If an assertion fails, execution stops at the violated contract.

If your result differs, compare inputs and types before rerunning.
Then explain the example from input to evidence in your own words.

### Worked example 2 — Represent cases as data

The same behavior can be checked across several boundaries. Predict first; then run the next cell.

In [ ]:
def is_valid_percentage(value: float) -> bool:
    return 0 <= value <= 100

cases = [(0, True), (100, True), (-0.1, False), (100.1, False)]
results = [(value, is_valid_percentage(value) == expected)
           for value, expected in cases]
results

**Expected observation:** `[(0, True), (100, True), (-0.1, True), (100.1, True)]`. Each tuple reports that its expectation matched.

## Debugging clinic

When evidence differs from your prediction, use this order:

1. Read the first failing assertion and reproduce that smallest case before scanning later failures.
2. Assert directly on useful values rather than wrapping them in an opaque helper returning only `True`/`False`.
3. Make fixtures create and clean up only state owned by the test.
4. If a test passes alone but fails in the suite, look for shared mutable state, order dependence, time, or randomness.

**Alternative to compare:** A doctest can suit tiny documentation examples, while pytest is clearer for fixtures, parametrization, exceptions, and larger behavior contracts.

**Boundary to test:** Empty input, exact bounds, malformed values, repeated calls, and cleanup after an exception reveal weak tests.

Do not move on merely because the cell runs. Explain which object,
branch, axis, row, or resource changed and why.

In [ ]:
# Example function
def add(a, b):
    return a + b

# In real projects, create tests/test_utils.py with:
# import pytest
# from mymodule import add
# @pytest.mark.parametrize('a,b,exp', [(1,2,3),(0,0,0),(-1,1,0)])
# def test_add(a,b,exp):
#     assert add(a,b) == exp

assert add(2,3) == 5


## Exercises and progressive hints

Each item is a complete mini-contract. Before writing code, copy its input,
expected behavior, constraints, and verification into your work cell. A
result is not complete merely because it “looks right”; run the stated
assertion or inspection and explain what it proves.

1. Write pytest tests for `safe_divide(numerator, denominator)` and `validate_email(text)`. **Coverage:** one ordinary result, negative values, zero denominator, one accepted bounded email example, and several rejected near misses. **Constraints:** name each behavior clearly and compare observable return values directly.
   **Verify:** run the test file from the repository root and confirm all intended cases are collected.

2. Add negative tests with `pytest.raises` for the narrow exception types documented by both functions. **Constraints:** keep only the expected failing call inside each context manager and match one stable, useful message fragment.
   **Verify:** temporarily change the expected exception once to observe a meaningful failure, then restore it.

### Additional mastery practice

Test observable contracts across normal, boundary, and invalid inputs. A good failure explains which behavior changed.

Continue with five new exercises. Record each prediction before running
code; these extend rather than replace the original practice above.

3. **Prediction:** Predict how pytest reports `assert actual == expected` compared with `assert check(actual)` when the values differ.
   **Progressive hint:** Direct comparisons usually produce more useful assertion introspection.
   **Verify:** Run both deliberately failing assertions once and compare pytest's messages; record which report exposes actual/expected values directly.
4. **Tracing:** Trace fixture setup, test execution, and teardown when the test passes and when it raises.
   **Progressive hint:** A yielding fixture resumes after `yield` for cleanup in both paths.
   **Verify:** Append events from setup, test, and teardown for pass and raise cases; assert teardown is the final event in both traces.
5. **Implementation:** Write parameterized tests for a slug function covering ordinary text, extra whitespace, punctuation, and empty text.
   **Progressive hint:** Each parameter row should communicate one behavior.
   **Verify:** Confirm pytest reports a separate named case for ordinary, whitespace, punctuation, and empty inputs and that all match the written slug contract.
6. **Debugging:** Repair a test whose `pytest.raises(Exception)` would accept unrelated bugs and whose protected block contains several operations.
   **Progressive hint:** Assert a narrow exception around one operation.
   **Verify:** Prove the narrow expected error passes, then trigger an unrelated error and confirm the test fails instead of accepting it.
7. **Edge case and explanation:** Test floating-point output and `NaN` correctly. Explain why direct equality is inappropriate for each.
   **Progressive hint:** Use `pytest.approx` for tolerance and `math.isnan` for NaN.
   **Verify:** Assert one computed float passes with `pytest.approx`, assert `math.isnan(value)` is true for NaN, and record that `value == value` is false.

Before opening the reference solution, write one sentence explaining
which contract or mental model each result confirms.

### Practice 1 — prediction, attempt, and evidence

**Contract reminder:** Write pytest tests for `safe_divide(numerator, denominator)` and `validate_email(text)`. **Coverage:** one ordinary result, negative values, zero denominator, one accepted bounded email example, and several rejected near misses. **Constraints:** name each behavior clearly and compare observable return values directly. **Verify:** run the test file from the repository root and confirm all intended cases are collected.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 1 — your work
# Short contract: Write pytest tests for `safe_divide(numerator, denominator)` and `validate_email(text)`. one ordinary result, negative values, zero denominator, one accepted bounded email examp...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 2 — prediction, attempt, and evidence

**Contract reminder:** Add negative tests with `pytest.raises` for the narrow exception types documented by both functions. **Constraints:** keep only the expected failing call inside each context manager and match one stable, useful message fragment. **Verify:** temporarily change the expected exception once to observe a meaningful failure, then restore it.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 2 — your work
# Short contract: Add negative tests with `pytest.raises` for the narrow exception types documented by both functions. keep only the expected failing call inside each context manager and match on...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 3 — prediction, attempt, and evidence

**Contract reminder:** **Prediction:** Predict how pytest reports `assert actual == expected` compared with `assert check(actual)` when the values differ. **Progressive hint:** Direct comparisons usually produce more useful assertion introspection. **Verify:** Run both deliberately failing assertions once and compare pytest's messages; record which report exposes actual/expected values directly.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 3 — your work
# Short contract: Predict how pytest reports `assert actual == expected` compared with `assert check(actual)` when the values differ. Direct comparisons usually produce more useful assertion intr...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 4 — prediction, attempt, and evidence

**Contract reminder:** **Tracing:** Trace fixture setup, test execution, and teardown when the test passes and when it raises. **Progressive hint:** A yielding fixture resumes after `yield` for cleanup in both paths. **Verify:** Append events from setup, test, and teardown for pass and raise cases; assert teardown is the final event in both traces.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 4 — your work
# Short contract: Trace fixture setup, test execution, and teardown when the test passes and when it raises. A yielding fixture resumes after `yield` for cleanup in both paths. Append events from...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 5 — prediction, attempt, and evidence

**Contract reminder:** **Implementation:** Write parameterized tests for a slug function covering ordinary text, extra whitespace, punctuation, and empty text. **Progressive hint:** Each parameter row should communicate one behavior. **Verify:** Confirm pytest reports a separate named case for ordinary, whitespace, punctuation, and empty inputs and that all match the written slug contract.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 5 — your work
# Short contract: Write parameterized tests for a slug function covering ordinary text, extra whitespace, punctuation, and empty text. Each parameter row should communicate one behavior. Confirm...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 6 — prediction, attempt, and evidence

**Contract reminder:** **Debugging:** Repair a test whose `pytest.raises(Exception)` would accept unrelated bugs and whose protected block contains several operations. **Progressive hint:** Assert a narrow exception around one operation. **Verify:** Prove the narrow expected error passes, then trigger an unrelated error and confirm the test fails instead of accepting it.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 6 — your work
# Short contract: Repair a test whose `pytest.raises(Exception)` would accept unrelated bugs and whose protected block contains several operations. Assert a narrow exception around one operation....
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 7 — prediction, attempt, and evidence

**Contract reminder:** **Edge case and explanation:** Test floating-point output and `NaN` correctly. Explain why direct equality is inappropriate for each. **Progressive hint:** Use `pytest.approx` for tolerance and `math.isnan` for NaN. **Verify:** Assert one computed float passes with `pytest.approx`, assert `math.isnan(value)` is true for NaN, and record that `value == value` is false.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 7 — your work
# Short contract: Test floating-point output and `NaN` correctly. Explain why direct equality is inappropriate for each. Use `pytest.approx` for tolerance and `math.isnan` for NaN. Assert one com...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):
